In [13]:
import os
import json
import orjson


def normalize_file_name(file_name):
    return file_name.replace('/', '_')


def keep_key(key):
    if key.endswith("acc"):
        return False
    
    if "sciq" in key:
        return False

    if "siqa" in key:
        return False

    return True


def get_slider_max(data):
    metrics = data[list(data.keys())[0]]
    metric_data = metrics[list(metrics.keys())[0]]
    samples = len(metric_data["x"])
    if samples < 20:
        return 10
    return 30


def create_index(data, traces, layout, default_window_size, default_metric):
    print(default_metric if default_metric else "None")
    files_data = {}
    index_files = {}
    for task_id, task_data in (data.items() if data else traces.items()):
        data_name = "data" if data else "traces"
        files_data[task_id] = {
            data_name: task_data,
            "layout": layout
        }
        index_files[task_id] = {
            "file": f"{normalize_file_name(task_id)}.json"
        }
    settings = {
        "slider": {
            "min": 0,
            "max": get_slider_max(data),
            "default": default_window_size,
        },
        "defaultMetric": default_metric
    } if data else {"slider": None}
    
    return files_data, index_files, settings
    
    

new_data = {}

for file_name in os.listdir('./data/plots'):
    if not file_name.endswith('.json'):
        continue
    with open(f'./data/plots/{file_name}', 'r') as file:
        old_data = orjson.loads(file.read())
    data = {key: value for key, value in old_data["data"].items() if keep_key(key)} if "data" in old_data else {}
    traces = {key: value for key, value in old_data["traces"].items()} if "traces" in old_data else {}
    default_window_size = old_data["defaultWindowSize"] if "defaultWindowSize" in old_data else None
    default_metric = old_data["defaultMetric"] if "defaultMetric" in old_data else None
    files_data, index_files, settings = create_index(data, traces, old_data["layout"], default_window_size, default_metric)
    # mkdir
    dir_name = file_name.split('.')[0]
    os.makedirs(f'./data/plots/{dir_name}', exist_ok=True)
    with open(f'./data/plots/{dir_name}/index.json', 'wb') as file:
        file.write(orjson.dumps({
            "files": index_files,
            "settings": settings,
        }))
    
    for metric_name, data in files_data.items():
        with open(f'./data/plots/{dir_name}/{normalize_file_name(metric_name)}.json', 'wb') as file:
            file.write(orjson.dumps(data))


